# Chapter 26 — Personalization with an inspectable evidence contract
Read `STUDY_CONTRACT.md` first. This notebook ships with empty outputs. Execute it yourself from this directory using Python 3.11+, then explain your own results in `ASSIGNMENT.md`'s format. The authoring assistant inspected code statically without executing the simulator or notebook.

Three distinct questions guide the exercise: does a frozen personal reference change held-out prediction; how does a **separate** randomized experiment distinguish risk from net utility; and when may an abstract software parameter be delivered? Nothing here is a neurological disease model or clinical controller.

In [ ]:
from collections import Counter
from pprint import pprint
from precision_studio import (
    make_fixture, build_rows, fit_risk, evaluate_risk, fit_effect_cells,
    compare_policies, snapshot, decide_gain, fault_cases, export_bundle,
)
fixture = make_fixture(seed=2601)
rows, baselines = build_rows(fixture)
pprint(Counter(p["cohort"] for p in fixture["people"]))
pprint(Counter(r["status"] for r in rows))
pprint(fixture["observations"][0])
pprint(fixture["references"]["d-001:18"])

## A–B. Freeze fitting before looking at final evaluation
List the eligible source fields at day 18. Why is the next-day reference unavailable then? Training uses only development people and mature labels by calendar cutoff 60. Validation and evaluation people may supply unlabeled days 0–9 for their own references; their outcomes must not fit either model.

Both models are shared fits with the same eligible rows. The raw model uses value and sleep; the personal model uses frozen z and sleep. This is a matched representation comparison, not a test of whether a raw model could operate without any reference. Inspect validation first. Record your final evaluation plan in the next markdown cell before running the evaluation cell.

In [ ]:
models = {
    "raw": fit_risk(rows, fixture["references"], personal=False),
    "personal": fit_risk(rows, fixture["references"], personal=True),
}
validation = {name: evaluate_risk(rows, fixture["references"], model, "validation")
              for name, model in models.items()}
pprint(validation)

**Record your evaluation plan here before proceeding.** State frozen model versions and comparisons, eligibility and coverage reporting, and how you will discuss dependence. Empty bins and one-class AUC are undefined. These reports supply no confidence interval or individual calibration. Changes made after evaluation inspection must be labeled exploratory.

In [ ]:
evaluation = {name: evaluate_risk(rows, fixture["references"], model, "evaluation")
              for name, model in models.items()}
pprint(evaluation)

## C. Trace the person and the measurement configuration
Person `e-004` has an eligible reference example at day 18 and a later firmware change. Person `e-003` supplies a contrasting physical-shift mechanism. Mechanisms can overlap across the fixture; these ID choices do not constitute a clinical classifier. Compare the frozen baseline, current source metadata, and later status reasons. A z score is a descriptive deviation, not a probability.

In [ ]:
person = next(p for p in fixture["people"] if p["pid"] == "e-004")
record = next(r for r in fixture["observations"] if r["pid"] == person["pid"] and r["day"] == 18)
baseline = baselines[person["pid"]]
current = snapshot(person, record, baseline)
pprint(baseline)
pprint(current)
pprint(baselines["e-000"])
for pid in ("e-004", "e-003"):
    pprint([(r["day"], r["source"]["firmware"], r["z"], r["status"], r["reason"])
            for r in rows if r["pid"] == pid and r["day"] in (18, 21, 22, 25)])

## Optional visual inspection
This cell may be skipped without changing any model or report. It plots recorded values alongside personal deviations in separate axes, with incompatible/ineligible snapshots omitted from the z series. Missing points are not zeros. A line shows the frozen mean, not a diagnosis. Firmware and physical change need source/context interpretation.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Optional plot skipped; inspect the tabulated records above.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    for pid in ("e-004", "e-003"):
        observed = [r for r in fixture["observations"] if r["pid"] == pid]
        eligible = [r for r in rows if r["pid"] == pid and r["status"] == "eligible"]
        axes[0].plot([r["day"] for r in observed], [r["value"] for r in observed], label=pid)
        axes[0].axhline(baselines[pid]["mean"], linestyle=":", alpha=.5)
        axes[1].scatter([r["day"] for r in eligible], [r["z"] for r in eligible], label=pid, s=16)
    axes[0].set_ylabel("Recorded value (arbitrary units)")
    axes[1].set_ylabel("Eligible personal z")
    axes[1].set_xlabel("Relative day")
    for ax in axes:
        ax.legend()
        ax.grid(alpha=.2)
    fig.tight_layout()
    plt.show()

## D. A separate randomized utility experiment
The supplied `baseline_risk` below is **not** a longitudinal predictor output. Fit group/burden cell contrasts only on development observations. Retain arm counts. At most 25% may be selected; positive-utility policies can select fewer. The oracle enters scoring and a labeled reference ranking only because the simulator defines both potential responses. Explain finite-sample error and why these cell averages are not observed individual effects.

In [ ]:
effect_cells = fit_effect_cells(fixture["randomized_trial"])
utility_comparison = compare_policies(fixture["randomized_trial"], effect_cells, fixture["oracle"])
pprint(effect_cells)
pprint(utility_comparison)

## E. Propose, permit, deliver
The bounded gain is an abstract software parameter, not a clinical setting. A default call must not deliver it. Compare the same eligible snapshot with explicit teaching permission. The Boolean flag implements no authentication service. An in-range number alone is not evidence of safe physical control.

In [ ]:
no_release = decide_gain(current)
released = decide_gain(current, release_allowed=True)
pprint(no_release)
pprint(released)

## F. Faults and the freeze latch
Inspect six independently altered versions of the same source. Preserve the returned state when simulating sequential decisions. The final call uses a later eligible snapshot to isolate the latch behavior; an eligible value is not evidence of requalification. The helper intentionally supplies no requalification/reset service.

In [ ]:
faults = fault_cases(person, record, baseline)
fault_decisions = {name: decide_gain(row, release_allowed=True) for name, row in faults.items()}
pprint({name: (row["status"], row["reason"]) for name, row in faults.items()})
firmware_state = fault_decisions["firmware"]["next_state"]
later_eligible = next(r for r in rows if r["pid"] == person["pid"] and r["day"] > 18 and r["status"] == "eligible")
still_frozen = decide_gain(later_eligible, state=firmware_state, release_allowed=True)
pprint(still_frozen)

## G. Export and interpret
Running this cell creates a new local output directory. The receipt hashes every other exported file but does not authenticate or make them immutable. Outcomes and oracle remain separate from observations. Submit your own observed results with the manifest and assignment interpretation. Explain where the evidence stops: no neurological validity, individual treatment effect, clinical calibration, physical safety, or prospective benefit is established.

In [ ]:
reports = {"validation": validation, "evaluation": evaluation,
           "effect_cells": effect_cells, "utility_comparison": utility_comparison}
decisions = {"no_release": no_release, "released": released,
             "faults": fault_decisions, "still_frozen": still_frozen}
destination = export_bundle(fixture, models, reports, decisions)
print(destination)